> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 历史学习课：用于理解问题，不取代当前H01/H02。窗口对比只针对已开放的三期，不要用它探索待封存的最终标签。

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import pandas as pd

DATA = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/olist-v2.zip').is_file()), Path.cwd()) / 'data' / 'olist-v2.zip'
assert DATA.exists(), '请在本notebook所在的olist_learning目录运行，旁边需要data/olist-v2.zip'
expected = '967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784'
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == expected
with ZipFile(DATA) as z:
    files = sorted(n for n in z.namelist() if n.endswith('.csv'))
    frames = {}
    for name in files:
        # 邮编作为代码读取，避免把代码当作连续数量；不填补、不删记录。
        with z.open(name) as f:
            frames[name] = pd.read_csv(f, dtype={
                'customer_zip_code_prefix': 'string',
                'seller_zip_code_prefix': 'string',
                'geolocation_zip_code_prefix': 'string'
            })
print('ZIP校验通过；CSV数量：', len(files))
print(pd.DataFrame([{'文件': n, '行数': len(frames[n]), '列数': frames[n].shape[1]}
                    for n in files]).to_string(index=False))


## 1. 逐张表列出全部字段：含义、读取类型、缺失和不同值数
本数据九张表合计52个字段位置，同名字段在不同表分别计数。CSV读取类型不等于业务类型：时间可能暂时读成object；编号不是可以求平均的测量值。
以下字段解释用于理解结构。少数时间字段的业务生成机制仍应核查来源，不凭名字推定预测时可用。


### customers：olist_customers_dataset.csv

**一行是什么：**订单侧客户资料；customer_id在本文件中每行唯一，不等于跨订单买家身份。

| 字段 | 理解用途 |
|---|---|
| customer_id | 连接订单的客户编号；候选键 |
| customer_unique_id | 跨订单识别买家的编号；本表不唯一 |
| customer_zip_code_prefix | 邮编前缀；代码而非数量 |
| customer_city | 客户城市 |
| customer_state | 客户州代码 |

In [ ]:
t = frames['olist_customers_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### orders：olist_orders_dataset.csv

**一行是什么：**一笔订单。

| 字段 | 理解用途 |
|---|---|
| order_id | 订单编号；候选键 |
| customer_id | 连接customers的编号 |
| order_status | 记录的订单状态 |
| order_purchase_timestamp | 下单时间 |
| order_approved_at | 批准时间 |
| order_delivered_carrier_date | 交给承运方的时间记录 |
| order_delivered_customer_date | 送达客户的时间记录 |
| order_estimated_delivery_date | 预计送达日期 |

In [ ]:
t = frames['olist_orders_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### order_items：olist_order_items_dataset.csv

**一行是什么：**订单内一条商品明细；一行不一定代表一种不同商品。

| 字段 | 理解用途 |
|---|---|
| order_id | 所属订单编号；复合键组成部分 |
| order_item_id | 订单内明细序号；复合键组成部分 |
| product_id | 商品编号 |
| seller_id | 卖家编号 |
| shipping_limit_date | 明细对应的发货截止时间字段；使用前应核查来源语义 |
| price | 该条明细商品价格 |
| freight_value | 该条明细记录的运费 |

In [ ]:
t = frames['olist_order_items_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### payments：olist_order_payments_dataset.csv

**一行是什么：**订单的一条支付记录；一单可以多条。

| 字段 | 理解用途 |
|---|---|
| order_id | 所属订单编号；复合键组成部分 |
| payment_sequential | 订单内支付序号；复合键组成部分 |
| payment_type | 支付方式 |
| payment_installments | 分期期数 |
| payment_value | 该条支付金额 |

In [ ]:
t = frames['olist_order_payments_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### reviews：olist_order_reviews_dataset.csv

**一行是什么：**与某订单关联的一条评价记录；review_id单列不能在此CSV中唯一识别行。

| 字段 | 理解用途 |
|---|---|
| review_id | 评价编号；实测不唯一 |
| order_id | 关联订单编号 |
| review_score | 评价分数；有序评分 |
| review_comment_title | 评价标题；文本缺失不自动等于负面评价 |
| review_comment_message | 评价正文 |
| review_creation_date | 评价创建日期字段；使用前核查业务含义 |
| review_answer_timestamp | 评价提交/回答时间字段 |

In [ ]:
t = frames['olist_order_reviews_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### products：olist_products_dataset.csv

**一行是什么：**一个商品的属性。

| 字段 | 理解用途 |
|---|---|
| product_id | 商品编号；候选键 |
| product_category_name | 原始品类名称 |
| product_name_lenght | 商品名长度；保留原始拼写 |
| product_description_lenght | 商品描述长度；保留原始拼写 |
| product_photos_qty | 商品图片数量 |
| product_weight_g | 重量，克 |
| product_length_cm | 长度，厘米 |
| product_height_cm | 高度，厘米 |
| product_width_cm | 宽度，厘米 |

In [ ]:
t = frames['olist_products_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### sellers：olist_sellers_dataset.csv

**一行是什么：**一个卖家的资料。

| 字段 | 理解用途 |
|---|---|
| seller_id | 卖家编号；候选键 |
| seller_zip_code_prefix | 卖家邮编前缀 |
| seller_city | 卖家城市 |
| seller_state | 卖家州代码 |

In [ ]:
t = frames['olist_sellers_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### geolocation：olist_geolocation_dataset.csv

**一行是什么：**一条邮编前缀相关的坐标记录；既不是每个邮编一行，也未证明每行唯一。

| 字段 | 理解用途 |
|---|---|
| geolocation_zip_code_prefix | 邮编前缀；实测不唯一 |
| geolocation_lat | 纬度 |
| geolocation_lng | 经度 |
| geolocation_city | 记录的城市名 |
| geolocation_state | 州代码 |

In [ ]:
t = frames['olist_geolocation_dataset.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


### translation：product_category_name_translation.csv

**一行是什么：**一个品类名称的英文对照。

| 字段 | 理解用途 |
|---|---|
| product_category_name | 原始品类名称；候选键 |
| product_category_name_english | 英文品类名称 |

In [ ]:
t = frames['product_category_name_translation.csv']
column_audit = pd.DataFrame({
    '字段': t.columns,
    '读取类型': [str(t[c].dtype) for c in t.columns],
    '缺失数': [int(t[c].isna().sum()) for c in t.columns],
    '不同非空值数': [int(t[c].nunique()) for c in t.columns]
})
print(column_audit.to_string(index=False))
print('整行重复数量：', t.duplicated().sum())


## 2. 主键不是看字段名猜出来的
主键：我们选定的、非空且唯一识别一行的键；可以由多列共同组成。
候选键：满足条件、可以被选作主键的字段组合。CSV没有数据库约束，本课先做“这份文件的候选键检查”，不声称已经获得完整业务模式。

检查顺序：有没有缺失 → 单列是否唯一 → 必要时检查复合键 → 结合一行的业务含义解释。
即使某列在一个样本里唯一，也不能单凭这一点认定它是业务上长期可靠的键。


In [ ]:
key_candidates = [
    ('orders', 'olist_orders_dataset.csv', ['order_id']),
    ('orders_customer_id', 'olist_orders_dataset.csv', ['customer_id']),
    ('customers', 'olist_customers_dataset.csv', ['customer_id']),
    ('customers_buyer_id', 'olist_customers_dataset.csv', ['customer_unique_id']),
    ('items_order_only', 'olist_order_items_dataset.csv', ['order_id']),
    ('items_composite', 'olist_order_items_dataset.csv', ['order_id','order_item_id']),
    ('payments_composite', 'olist_order_payments_dataset.csv', ['order_id','payment_sequential']),
    ('reviews_id_only', 'olist_order_reviews_dataset.csv', ['review_id']),
    ('reviews_composite', 'olist_order_reviews_dataset.csv', ['review_id','order_id']),
    ('products', 'olist_products_dataset.csv', ['product_id']),
    ('sellers', 'olist_sellers_dataset.csv', ['seller_id']),
    ('translation', 'product_category_name_translation.csv', ['product_category_name']),
    ('geolocation_prefix', 'olist_geolocation_dataset.csv', ['geolocation_zip_code_prefix'])
]
key_rows=[]
for label, filename, keys in key_candidates:
    t=frames[filename]
    nulls=int(t[keys].isna().any(axis=1).sum())
    duplicates=int(t.duplicated(keys).sum())
    key_rows.append({'检查': label, '字段组合': ' + '.join(keys), '空键行数': nulls,
                     '重复键行数_超出首行': duplicates, '本文件非空唯一': nulls==0 and duplicates==0})
print(pd.DataFrame(key_rows).to_string(index=False))


**结果解释：**
- orders.order_id、customers.customer_id、products.product_id、sellers.seller_id和品类对照表的原始品类名，都在这份文件中非空且唯一。
- 商品明细需要(order_id,order_item_id)；支付需要(order_id,payment_sequential)。序号只在同一订单内识别明细。
- review_id存在814个超出首次出现的重复；(review_id,order_id)组合无重复，可作为本文件中的候选复合键，但仍需了解评价生成机制。不能删掉所有重复review_id而不检查它关联的订单。
- geolocation的邮编前缀不是主键；全表还有261831个超出首次出现的整行重复。先记录，不在不了解用途时直接决定去重或取平均坐标。
- orders.customer_id在本文件也唯一，因此orders与customers按customer_id连接是观测到的1:1；不意味着一个自然买家只会下一个订单。跨订单买家由customer_unique_id关联。


## 3. 外键与基数：逻辑上能连，不等于随便merge
外键是子表中引用父表键的列。这里是CSV，所以检查的是逻辑引用关系，不是数据库已经强制执行的外键约束。
检验：父端是否唯一？子端有没有空键？非空键在父端能否找到？每个父记录可以对应几条子记录？

结构地图：

```text
隐含买家实体(customer_unique_id)
  └─ customers -- customer_id --> orders
        │                          ├─ order_items -- product_id --> products -- category --> translation
        │                          │       └─ seller_id --> sellers
        │                          ├─ payments
        │                          └─ reviews
        └─ customer_zip_code_prefix ─┐
                         sellers.zip ├─ geolocation（邮编不是唯一，不能当普通唯一维表）
```

这里的orders到items/payments/reviews是一对多（有些订单无子记录）。items到products/sellers是多对一。


In [ ]:
relations = [
    ('orders→customers','olist_orders_dataset.csv','customer_id','olist_customers_dataset.csv','customer_id'),
    ('items→orders','olist_order_items_dataset.csv','order_id','olist_orders_dataset.csv','order_id'),
    ('payments→orders','olist_order_payments_dataset.csv','order_id','olist_orders_dataset.csv','order_id'),
    ('reviews→orders','olist_order_reviews_dataset.csv','order_id','olist_orders_dataset.csv','order_id'),
    ('items→products','olist_order_items_dataset.csv','product_id','olist_products_dataset.csv','product_id'),
    ('items→sellers','olist_order_items_dataset.csv','seller_id','olist_sellers_dataset.csv','seller_id'),
    ('products→translation','olist_products_dataset.csv','product_category_name','product_category_name_translation.csv','product_category_name'),
    ('customers→geo','olist_customers_dataset.csv','customer_zip_code_prefix','olist_geolocation_dataset.csv','geolocation_zip_code_prefix'),
    ('sellers→geo','olist_sellers_dataset.csv','seller_zip_code_prefix','olist_geolocation_dataset.csv','geolocation_zip_code_prefix')
]
relation_rows=[]
for label,cn,ck,pn,pk in relations:
    child,parent=frames[cn],frames[pn]
    orphan=child[ck].notna() & ~child[ck].isin(parent[pk].dropna())
    parent_values=parent[pk].dropna().drop_duplicates()
    relation_rows.append({'关系': label,'父端键唯一': parent[pk].is_unique,
        '子端空键行数': int(child[ck].isna().sum()),'非空但父端找不到_行数': int(orphan.sum()),
        '父端找不到_不同键数': int(child.loc[orphan,ck].nunique()),
        '无子记录的父键数': int((~parent_values.isin(child[ck].dropna())).sum())})
print(pd.DataFrame(relation_rows).to_string(index=False))


**怎样解释这些关联？**
- 明细、支付、评价的非空order_id都能在订单表找到；但反方向并非每笔订单都有子记录。
- 775笔订单没有商品明细，1笔没有支付记录，768笔没有评价。是否排除它们，必须等业务问题确定，不把无评价自动当作差评。
- 商品品类有610个空值；另有13个商品的非空品类在对照表找不到。缺原始品类与缺英文翻译不是同一种缺失。
- 278条客户资料的邮编前缀和7个卖家的邮编前缀找不到地理记录。邮编映射也不是100%覆盖。
- geo父端键不唯一，因此不能直接用many_to_one验证通过。需根据未来用途决定邮编层级的坐标或其他地理映射，不能默认每个前缀对应唯一城市、唯一坐标。


In [ ]:
from IPython.display import SVG, display
from html import escape

# 每张表的位置、名称和关键字段
tables = [
    (120,  40, "geolocation", [
        "ZIP prefix is NOT unique",
        "Multiple coordinate records"
    ]),
    (560,  40, "customers", [
        "Key*: customer_id",
        "Buyer: customer_unique_id"
    ]),
    (560, 220, "orders", [
        "Key*: order_id",
        "Link: customer_id"
    ]),
    (120, 400, "payments", [
        "Key*: order_id",
        "    + payment_sequential"
    ]),
    (560, 400, "order_items", [
        "Key*: order_id + order_item_id",
        "Links: product_id, seller_id"
    ]),
    (1000, 400, "reviews", [
        "Key*: review_id + order_id",
        "review_id alone repeats"
    ]),
    (120, 600, "category_translation", [
        "Key*: product_category_name",
        "product_category_name_english"
    ]),
    (560, 600, "products", [
        "Key*: product_id",
        "Link: product_category_name"
    ]),
    (1000, 600, "sellers", [
        "Key*: seller_id",
        "seller_zip_code_prefix"
    ]),
]

# 连线路径、标签及标签位置
# 最后一项 True 表示虚线：邮编关联不能假定一对一
links = [
    ("560,95 370,95",
     "customer ZIP prefix", 465, 78, True),

    ("685,150 685,220",
     "customer_id", 755, 190, False),

    ("560,275 245,275 245,400",
     "order_id", 405, 262, False),

    ("685,330 685,400",
     "order_id", 745, 370, False),

    ("810,275 1125,275 1125,400",
     "order_id", 970, 262, False),

    ("685,510 685,600",
     "product_id", 755, 560, False),

    ("810,485 900,485 900,565 1125,565 1125,600",
     "seller_id", 1020, 550, False),

    ("560,655 370,655",
     "product_category_name", 465, 640, False),

    ("1125,710 1125,780 50,780 50,95 120,95",
     "seller ZIP prefix → geolocation ZIP prefix",
     650, 800, True),
]

parts = [
    """
    <svg xmlns="http://www.w3.org/2000/svg"
         width="1350" height="860"
         viewBox="0 0 1350 860"
         style="max-width:100%; height:auto; background:white;">
    <rect width="1350" height="860" fill="#ffffff"/>
    """
]

# 先画连接线
for points, label, lx, ly, dashed in links:
    dash = 'stroke-dasharray="8 6"' if dashed else ""
    color = "#d97706" if dashed else "#64748b"

    parts.append(f"""
        <polyline points="{points}"
                  fill="none"
                  stroke="{color}"
                  stroke-width="2"
                  {dash}/>
    """)

# 再画表框与字段
for x, y, title, fields in tables:
    parts.append(f"""
        <rect x="{x}" y="{y}" width="250" height="110"
              rx="10" fill="#f8fafc"
              stroke="#334155" stroke-width="1.5"/>

        <path d="M {x} {y+38} H {x+250}"
              stroke="#cbd5e1"/>

        <text x="{x+125}" y="{y+26}"
              text-anchor="middle"
              font-family="Arial, sans-serif"
              font-size="17" font-weight="bold"
              fill="#0f172a">{escape(title)}</text>
    """)

    for i, field in enumerate(fields):
        parts.append(f"""
            <text x="{x+12}" y="{y+63+i*23}"
                  font-family="Consolas, monospace"
                  font-size="12"
                  fill="#334155">{escape(field)}</text>
        """)

# 添加连接字段标签
for points, label, lx, ly, dashed in links:
    parts.append(f"""
        <text x="{lx}" y="{ly}"
              text-anchor="middle"
              font-family="Arial, sans-serif"
              font-size="12"
              fill="#475569"
              style="paint-order:stroke; stroke:white; stroke-width:5;">
            {escape(label)}
        </text>
    """)

parts.append("""
    <text x="120" y="835"
          font-family="Arial, sans-serif"
          font-size="12" fill="#475569">
        Key*: candidate key checked in this CSV.
        Solid lines: ID/category links.
        Dashed lines: non-unique ZIP-prefix matching.
    </text>
    </svg>
""")

display(SVG("".join(parts)))

In [ ]:

from IPython.display import SVG, display
from html import escape

# ==============================
# 1. 基础设置
# ==============================

FONT = "'Microsoft YaHei', 'Noto Sans CJK SC', Arial, sans-serif"

# x、y、表名、中文解释、关键字段、主题色
nodes = [
    (60, 220, "geolocation", "地理记录 · 邮编前缀不唯一",
     ["geolocation_zip_code_prefix", "lat / lng / city / state"], "#D97706"),

    (470, 220, "customers", "客户资料 · 跨订单识别买家",
     ["Key*: customer_id", "Buyer: customer_unique_id"], "#2563EB"),

    (470, 400, "orders", "订单 · 一行一笔订单",
     ["Key*: order_id", "Link: customer_id"], "#0891B2"),

    (60, 580, "payments", "支付 · 一单可能多条记录",
     ["Key*: order_id", "    + payment_sequential"], "#64748B"),

    (470, 580, "order_items", "商品明细 · 一单可能多条明细",
     ["Key*: order_id + order_item_id",
      "Links: product_id / seller_id"], "#7C3AED"),

    (880, 580, "reviews", "评价 · 一单可能多条记录",
     ["Key*: review_id + order_id",
      "review_id 单列不唯一"], "#64748B"),

    (60, 770, "translation", "品类名称对照",
     ["Key*: product_category_name",
      "product_category_name_english"], "#059669"),

    (470, 770, "products", "商品属性 · 一行一个商品",
     ["Key*: product_id",
      "Link: product_category_name"], "#059669"),

    (880, 770, "sellers", "卖家资料 · 一行一个卖家",
     ["Key*: seller_id",
      "seller_zip_code_prefix"], "#059669"),
]


# ==============================
# 2. 定义表之间的连接关系
# ==============================
# cardinality:
# 1:1 = 一对一
# 1:N = 一对多
# 非唯一匹配 = 邮编前缀不唯一，不能视为严格的一对多外键关系
#
# 关系标签统一按「主表 -> 明细表」方向理解。

edges = [

    # 隐含买家 -> customers
    {
        "points": "600,170 600,220",
        "field_label": "customer_unique_id",
        "field_x": 740, "field_y": 199,
        "postal": False,
        "cardinality": "1:N",
        "card_x": 600, "card_y": 195
    },

    # customers <-> geolocation
    {
        "points": "470,275 320,275",
        "field_label": "客户邮编前缀",
        "field_x": 395, "field_y": 258,
        "postal": True,
        "cardinality": "非唯一匹配",
        "card_x": 395, "card_y": 300
    },

    # customers -> orders
    {
        "points": "600,330 600,400",
        "field_label": "customer_id",
        "field_x": 680, "field_y": 370,
        "postal": False,
        "cardinality": "1:1",
        "card_x": 600, "card_y": 370
    },

    # orders -> payments
    {
        "points": "470,455 190,455 190,580",
        "field_label": "order_id",
        "field_x": 330, "field_y": 440,
        "postal": False,
        "cardinality": "1:N",
        "card_x": 190, "card_y": 520
    },

    # orders -> order_items
    {
        "points": "600,510 600,580",
        "field_label": "order_id",
        "field_x": 670, "field_y": 550,
        "postal": False,
        "cardinality": "1:N",
        "card_x": 600, "card_y": 550
    },

    # orders -> reviews
    {
        "points": "730,455 1010,455 1010,580",
        "field_label": "order_id",
        "field_x": 870, "field_y": 440,
        "postal": False,
        "cardinality": "1:N",
        "card_x": 1010, "card_y": 520
    },

    # products -> order_items
    {
        "points": "600,690 600,770",
        "field_label": "product_id",
        "field_x": 680, "field_y": 735,
        "postal": False,
        "cardinality": "1:N",
        "card_x": 600, "card_y": 735
    },

    # sellers -> order_items
    {
        "points": "730,660 800,660 800,730 1010,730 1010,770",
        "field_label": "seller_id",
        "field_x": 910, "field_y": 715,
        "postal": False,
        "cardinality": "1:N",
        "card_x": 850, "card_y": 755
    },

    # translation -> products
    {
        "points": "470,825 320,825",
        "field_label": "品类名称",
        "field_x": 395, "field_y": 810,
        "postal": False,
        "cardinality": "1:N",
        "card_x": 395, "card_y": 850
    },

    # sellers <-> geolocation
    {
        "points": "1010,880 1010,925 25,925 25,275 60,275",
        "field_label": "卖家邮编前缀 ↔ 地理表邮编前缀",
        "field_x": 600, "field_y": 947,
        "postal": True,
        "cardinality": "非唯一匹配",
        "card_x": 130, "card_y": 905
    },
]


# ==============================
# 3. 创建 SVG 画布
# ==============================

svg = [f"""
<svg xmlns="http://www.w3.org/2000/svg"
     width="1200" height="1060"
     viewBox="0 0 1200 1060"
     style="max-width:100%; height:auto; background:#F8FAFC">

  <defs>
    <filter id="shadow" x="-15%" y="-15%"
            width="140%" height="150%">
      <feDropShadow dx="0" dy="3" stdDeviation="4"
                    flood-color="#0F172A"
                    flood-opacity="0.08"/>
    </filter>
  </defs>

  <rect width="1200" height="1060"
        rx="18" fill="#F8FAFC"/>

  <text x="600" y="40"
        text-anchor="middle"
        font-family="{FONT}"
        font-size="26"
        font-weight="bold"
        fill="#0F172A">
    Olist 九张表 · 数据关系图
  </text>

  <text x="600" y="68"
        text-anchor="middle"
        font-family="{FONT}"
        font-size="14"
        fill="#64748B">
    先理解关联与粒度，再决定如何汇总和连接
  </text>
""" ]


# ==============================
# 4. 绘制所有连接线
# ==============================

for edge in edges:

    points = edge["points"]
    postal = edge["postal"]

    color = "#D97706" if postal else "#94A3B8"

    dash = (
        'stroke-dasharray="7 6"'
        if postal else ""
    )

    svg.append(f"""
      <polyline
          points="{points}"
          fill="none"
          stroke="{color}"
          stroke-width="2"
          {dash}
          stroke-linejoin="round"/>
    """)


# ==============================
# 5. 绘制隐含买家实体
# ==============================

svg.append(f"""
  <rect x="470" y="100"
        width="260" height="70"
        rx="12"
        fill="#EFF6FF"
        stroke="#2563EB"
        stroke-width="1.5"
        stroke-dasharray="6 4"/>

  <text x="600" y="127"
        text-anchor="middle"
        font-family="{FONT}"
        font-size="17"
        font-weight="bold"
        fill="#1D4ED8">
    买家 · 隐含实体
  </text>

  <text x="600" y="151"
        text-anchor="middle"
        font-family="{FONT}"
        font-size="12"
        fill="#475569">
    通过编号识别，不是第十张 CSV
  </text>
""")


# ==============================
# 6. 绘制九张表
# ==============================

for x, y, title, subtitle, fields, color in nodes:

    svg.append(f"""
      <g filter="url(#shadow)">

        <rect x="{x}" y="{y}"
              width="260" height="110"
              rx="12"
              fill="white"
              stroke="#E2E8F0"/>

        <rect x="{x}" y="{y+12}"
              width="5" height="86"
              rx="2"
              fill="{color}"/>

      </g>

      <text x="{x+18}" y="{y+28}"
            font-family="{FONT}"
            font-size="19"
            font-weight="bold"
            fill="{color}">
        {escape(title)}
      </text>

      <text x="{x+18}" y="{y+50}"
            font-family="{FONT}"
            font-size="12"
            fill="#64748B">
        {escape(subtitle)}
      </text>
    """)

    # 表中的关键字段
    for j, field in enumerate(fields):

        svg.append(f"""
          <text x="{x+18}"
                y="{y+76+j*19}"
                font-family="Consolas, {FONT}"
                font-size="12"
                fill="#334155">
            {escape(field)}
          </text>
        """)


# ==============================
# 7. 绘制连接字段标签
# ==============================

for edge in edges:

    label = edge["field_label"]

    lx = edge["field_x"]
    ly = edge["field_y"]

    postal = edge["postal"]

    field_color = (
        "#B45309" if postal else "#475569"
    )

    svg.append(f"""
      <text x="{lx}" y="{ly}"
            text-anchor="middle"
            font-family="{FONT}"
            font-size="13"
            fill="{field_color}"
            style="paint-order:stroke;
                   stroke:#F8FAFC;
                   stroke-width:6;">
        {escape(label)}
      </text>
    """)


# ==============================
# 8. 新增：绘制 1:1 / 1:N 关系标签
# ==============================

for edge in edges:

    card = edge["cardinality"]

    cx = edge["card_x"]
    cy = edge["card_y"]

    postal = edge["postal"]

    # 根据标签自动调整背景框宽度
    card_w = 88 if card == "非唯一匹配" else 48
    card_h = 23

    # 颜色区分
    if postal:
        card_color = "#B45309"
        bg_color = "#FFF7ED"
    elif card == "1:1":
        card_color = "#1D4ED8"
        bg_color = "#EFF6FF"
    else:
        card_color = "#334155"
        bg_color = "#F1F5F9"

    font_size = 11 if postal else 13

    svg.append(f"""
      <rect
          x="{cx-card_w/2}"
          y="{cy-card_h/2}"
          width="{card_w}"
          height="{card_h}"
          rx="6"
          fill="{bg_color}"
          stroke="{card_color}"
          stroke-width="1"/>

      <text
          x="{cx}"
          y="{cy+4}"
          text-anchor="middle"
          font-family="{FONT}"
          font-size="{font_size}"
          font-weight="bold"
          fill="{card_color}">
        {escape(card)}
      </text>
    """)


# ==============================
# 9. 图例与说明
# ==============================

svg.append(f"""

  <text x="600" y="980"
        text-anchor="middle"
        font-family="{FONT}"
        font-size="13"
        fill="#475569">
    实线：编号或品类关联 ｜ 橙色虚线：非唯一邮编匹配 ｜ Key*：数据中验证的候选键
  </text>

  <text x="600" y="1005"
        text-anchor="middle"
        font-family="{FONT}"
        font-size="13"
        font-weight="bold"
        fill="#334155">
    1:1 = 一对一 ｜ 1:N = 一对多 ｜ 非唯一匹配 = 不保证唯一对应
  </text>

  <text x="600" y="1032"
        text-anchor="middle"
        font-family="{FONT}"
        font-size="11"
        fill="#64748B">
    关系标签按主表 → 明细表方向理解；customers → orders 的 1:1 是本数据集的记录结构
  </text>

</svg>
""")


# ==============================
# 10. 展示最终图像
# ==============================

display(SVG("".join(svg)))


In [ ]:

import pandas as pd
from IPython.display import display

# Olist 九张表的主键与数据粒度
data = [
    ["customers", "customer_id", "单字段", "一条客户记录"],
    ["orders", "order_id", "单字段", "一笔订单"],
    ["order_items", "order_id + order_item_id", "复合键", "一条订单商品明细"],
    ["payments", "order_id + payment_sequential", "复合键", "一条支付记录"],
    ["reviews", "review_id + order_id", "复合候选键", "一条评价记录"],
    ["products", "product_id", "单字段", "一件商品"],
    ["sellers", "seller_id", "单字段", "一个卖家"],
    ["translation", "product_category_name", "单字段", "一种商品类别的翻译"],
    ["geolocation", "无明确唯一主键", "—", "一条地理位置记录"],
]

columns = [
    "表名",
    "主键 / 候选键（Key）",
    "类型",
    "一行代表什么"
]

df = pd.DataFrame(data, columns=columns)

# 美化表格
styled_df = (
    df.style
    .set_caption("Olist 九张表 · Primary Keys & Data Granularity")
    .set_properties(**{
        "text-align": "center",
        "padding": "12px",
        "font-size": "13px",
        "border": "1px solid #E2E8F0"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#2563EB"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "12px")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "18px"),
                ("font-weight", "bold"),
                ("color", "#1E3A8A"),
                ("padding", "15px")
            ]
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [
                ("background-color", "#EFF6FF")
            ]
        },
        {
            "selector": "tbody tr:hover",
            "props": [
                ("background-color", "#DBEAFE")
            ]
        }
    ])
    .hide(axis="index")
)

display(styled_df)

# 可选：保存为 Excel
# 公开学习版保留屏幕表格；可选Excel导出留在私有原件。


## 4. 真正做一次小范围连接：为什么九表直接拼接会错？
先选一笔同时有2条商品明细、2条支付的真实订单。在原始粒度按order_id直接连接，两边会两两组合成4行。
这里演示错误与核查，不对全数据做错误大连接。


In [ ]:
items = frames['olist_order_items_dataset.csv']
payments = frames['olist_order_payments_dataset.csv']
ic=items.groupby('order_id').size()
pc=payments.groupby('order_id').size()
sizes=pd.concat([ic.rename('item_rows'),pc.rename('payment_rows')],axis=1).fillna(0)
example_id=sizes.loc[sizes['item_rows'].eq(2)&sizes['payment_rows'].eq(2)].index[0]
one_items=items.loc[items['order_id'].eq(example_id)]
one_payments=payments.loc[payments['order_id'].eq(example_id)]
bad=one_items.merge(one_payments,on='order_id',validate='many_to_many')
print('订单编号：',example_id)
print('原始明细：')
print(one_items[['order_item_id','price','freight_value']].to_string(index=False))
print('原始支付：')
print(one_payments[['payment_sequential','payment_type','payment_value']].to_string(index=False))
print('直接拼接后的组合：')
print(bad[['order_item_id','payment_sequential','price','payment_value']].to_string(index=False))
print('拼接行数：',len(bad))
print('商品金额原始合计：',round(one_items['price'].sum(),2),
      '拼接后错误合计：',round(bad['price'].sum(),2))
print('支付原始合计：',round(one_payments['payment_value'].sum(),2),
      '拼接后错误合计：',round(bad['payment_value'].sum(),2))

# 如果目标是一行一订单，两边先各自汇总成一行，再连接。
item_order=one_items.groupby('order_id').agg(goods_amount=('price','sum'),freight=('freight_value','sum'))
payment_order=one_payments.groupby('order_id').agg(payment_amount=('payment_value','sum'))
good=item_order.join(payment_order,validate='one_to_one')
print('演示订单层级的正确汇总：')
print(good.round(2).to_string())
assert len(good)==1
assert abs(good['goods_amount'].iloc[0]-one_items['price'].sum())<1e-9


**结果解释：** 这笔订单商品金额298.90，支付375.73；商品金额加明细运费76.83恰好等于支付金额。这只是本例核对成功，不是对全部订单的结论。
直接拼接会把商品金额误算为597.80、支付误算为751.46。
`validate='many_to_many'`并不能防止这种膨胀，只是允许多对多。因此“merge没报错”不等于分析正确。
若未来分析粒度是订单，可先汇总明细、支付、评价后分别连订单；若分析商品明细，订单级金额不能在每条明细重复求和。
客户层级还需要先判断跨订单买家，再汇总；今天只理解，不提前决定最终客户资格。


连接后，每条商品明细和每条支付记录都出现了两次。再对这4行求和，就把原来的金额算了两遍。

你使用的连接条件只有：明细表.merge(支付表, on="order_id")
只要两行的 order_id 相同，就把它们配在一起。商品明细1，可以匹配支付记录2，也可以匹配支付记录1。并没有多卖商品，只是连接产生了重复出现的价格。




## 5. 与数据库1NF、2NF、3NF有什么联系？
你提到的通常写作1NF、2NF、3NF（第一、第二、第三范式）。思路相关，但不需要把九张现成表重新机械拆一遍。

函数依赖A→B：在业务定义中，一个A值决定一个B值。数据检查可以发现反例，样本恰好没有反例不等于证明业务上永远成立。

### 1NF：按单值字段组织记录
没有把“一张订单的所有商品列表”塞进一个重复组列，而是用order_items逐条记录，便于关联与统计。长评论作为一个文本值本身不违反原子性。不要用“列中有逗号”判断违反1NF。

### 2NF：非主属性不依赖复合候选键的一部分
若以(order_id,order_item_id)作为明细键，却把order_purchase_timestamp反复存进明细表：订单时间只依赖order_id，属于部分依赖的典型例子。把订单属性放orders、明细属性放items有助于避免这类冗余。

### 3NF：去除不合适的传递依赖（正式定义：每个非平凡函数依赖X→A，X是超键或A是主属性）
例如把英文品类名存进每个商品：product_id→原始品类→英文品类，英文名称会随同一品类重复。独立的品类对照表可以避免这种冗余。
仅凭这些例子不能认证九张表全部满足3NF；还需完整候选键、业务函数依赖和来源说明。geo和reviews的键问题尤其需要先处理。

### 数据库存储与分析宽表不同
规范化服务于减少冗余和维护一致性；建模特征表常常有意汇总/反规范化成每个分析对象一行。不是先把九表全merge，就等于完成规范化或特征工程。


## 6. 全时期时间检查：按每种事件分别检查，不先筛订单
时间列先转换并报告解析失败，不因缺失立刻删行。下单、批准、送达、预计送达是不同事件；它们的最大日期不能互相替代。


In [ ]:
date_specs = {
    'olist_orders_dataset.csv': ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date'],
    'olist_order_items_dataset.csv': ['shipping_limit_date'],
    'olist_order_reviews_dataset.csv': ['review_creation_date','review_answer_timestamp']
}
date_rows=[]
parsed_dates={}
for filename, cols in date_specs.items():
    for col in cols:
        original=frames[filename][col]
        parsed=pd.to_datetime(original,errors='coerce')
        parsed_dates[(filename,col)]=parsed
        date_rows.append({'表': filename,'字段': col,'最早': str(parsed.min()),'最晚': str(parsed.max()),
                          '原始缺失': int(original.isna().sum()),
                          '非空但解析失败': int((original.notna()&parsed.isna()).sum())})
print(pd.DataFrame(date_rows).to_string(index=False))
p=parsed_dates[('olist_orders_dataset.csv','order_purchase_timestamp')]
a=parsed_dates[('olist_orders_dataset.csv','order_approved_at')]
months=pd.period_range(p.min().to_period('M'),p.max().to_period('M'),freq='M')
monthly=pd.DataFrame({'原始下单数':p.groupby(p.dt.to_period('M')).size(),
                       '有批准时间的订单数_按批准月':a.groupby(a.dt.to_period('M')).size()}).reindex(months).fillna(0).astype(int)
print('全部月份：')
print(monthly.to_string())


**结果解释：** 下单记录覆盖2016-09至2018-10，但不同事件日期的边界不同。2016年末、2018年末的月度记录稀疏，是数据覆盖待核查事项；本课不据此直接选择训练或测试时期。
预计送达可以晚于最后下单日，因为它是预计日期，不代表那一天有真实新订单。
月度记录为0只说明本文件没观察到该事件，不足以证明现实中没有业务。
时间先后异常可作为核查线索，不应未经来源核实就批量删除。尤其预计日期常为零点，应先决定按日期还是精确时间比较。


## 7. 质量问题怎样记录，而不是自动清洗？
每项记录“发现、可能影响、待核查、拟处理方式及理由”。目前先标记待定。

| 发现 | 可能影响 | 现在做什么 |
|---|---|---|
| 主订单无商品/支付/评价子记录 | 不同分析的可用样本不同 | 记录覆盖，不默认填0或删除 |
| review_id重复、每单可多评价 | 直接连接会重复订单 | 检查来源，未来确定评价聚合方式 |
| 地理前缀多行且有整行重复 | 连接膨胀，不能当唯一维表 | 标记；有地理用途时再定映射 |
| 品类缺失、翻译缺失 | 品类画像覆盖不全 | 区分两类缺失，保留来源信息 |
| 时间字段缺失及边界稀疏 | 事件定义、观察期受限制 | 先审查，不凭最大日期判完整 |
| 数值极端值 | 可能是真实大订单或错误 | 用分布与原始记录核查，不机械删尾部 |

统计字段之间的相关性属于后续EDA。先明确哪些是编号、类别、时间、数量，再选适合的比较；不要把ID编码后对52个字段盲算相关系数。

## 8. 本课交付：你应该能画出的结构与下一步
1. 给出九表各自粒度和52个字段的用途。
2. 解释主键/复合键，并指出reviews与geo不能按名字假定主键。主键就是一个属性可以唯一确认一行，复合键就是几个属性可以唯一确认一行，reviews与geo不能按名字假定主键为什么？同一个邮编前缀可以对应多条地理记录；要求非空且唯一。知道邮编前缀12345，还不能确定是哪一行。
因此：
能用来匹配，不代表能唯一识别一行。它可以由一列组成，也可以由多列组成。
3. 画出关系、写出连接键、说明1:1/1:N/多对多风险。容易一个数据重复2行计算多次导致数值加倍
4. 解释2条明细×2条支付为何变4行，说明何时先汇总。因为一条明细可能对应2条支付2条明细就4条支付了，就有数据重复2次了，当1：N的时候就要先汇总吗？什么时候才可以拼接呢所有数据都1：1：1的时候吗
5. 记录时间范围与质量问题，但不把尚未确定的业务排除规则当成清洗标准。

做到这些，再进入业务定义：服务谁 → 支持什么决定 → 分析单位 → 预测时点 → 结果事件及期限。
如果选复购，再定义什么叫购买/再次购买；如果选配送或评价，就定义相应目标，不预设复购。
随后按预测时点检查可用信息和标签覆盖，设计训练/验证/测试，再拟合预处理与模型，避免让未来信息参与训练。

现在先复算第4节的真实订单连接。你不需要先认同任何购买资格或90天窗口，也能验证自己是否真的理解表关系。


各表的粒度”就是：这张表的一行代表什么
你对“2条明细×2条支付”的解释，需要改这一句。

你说：
一条明细可能对应2条支付，2条明细就4条支付。

更准确的是：
连接条件只有 order_id。两条明细和两条支付的订单号都相同，所以每条明细都匹配到了两条支付，形成4种组合。

什么时候表格可以链接，不是只有所有表都1:1才可以连接。 判断方法是：
我希望结果的一行代表什么？连接后还能保持这个含义吗？哪些金额或数量会被重复展示，后面能不能直接求和？

连接之前确定粒度；连接之后重新检查粒度、行数和金额是否仍然正确。

这正是你以后独立分析数据时需要反复使用的思路：先观察，再提出结构问题，用代码验证，最后记录发现。